# Lecture  Experimental Control 1

- This lesson will review some of the fundamental programming methods required to develop a controlled experiment for Psychological/Neuroscience research

- We will use an auditory experiment as an example but the fundamental methods discussed of controlling the experiment and managing the data explained here will apply to any kind of experiment

- Any experiment has two properties. 
* There are some experimental **conditions** that have different stimuli (and/or task instructions) presented
* There is a **response** obtained from the subject.  

The main issues discussed here are:
* randomization
* obtaining responses from a participant

Two other issues we will discuss more later
* organizing behavioral data using Pandas DataFrames
* saving the data to a file 

### Randomization 

For the purpose of experimental control, we need to start the random number generator. 
The **seed** of the random number generator is very important here. 
If we never change the seed we will always run the exact sequence of the experiment on every participant. 
We should always use a different seed on different participants.  

In [1]:
import numpy as np 
from numpy import random 
import sounddevice as sd
from matplotlib import pyplot as plt
rng = random.default_rng(seed = 1111)

In [2]:
from sound import pop_scales
from sound import make_tone
from sound import play_sound
scales = pop_scales()

In [3]:
# fixed facts about the sounds we want to make functions
duration = 0.5 # length of the sound in seconds. 
volume = 0.1   # DO NOT MAKE VOLUME LARGER THAN 0.5 

### The A-B Task

- Let's consider a task where the subject has to identify a note played as A (440 Hz) or B (494 Hz).  On each trial, we will present one of the notes, and the subject's task is to respond with A or B on the keyboard. 

First let's make the two notes 

In [4]:
duration = 0.5 # length of the sound. 
f_A = scales['A']        # base frequency of the sinewave in Hz
A = make_tone(f_A,duration)
f_B = scales['B']        # base frequency of the sinewave in Hz
B = make_tone(f_B,duration)


### Randomization  

- To make an experiment that is well controlled, psychologists make use of randomization. 
- For example, if we want to carry out an experiment with different types of emotional stimuli (e.g., words with negative or positive affect), we usually want to present an experimental **condition** (negative or positive) chosen at random on each trial of the experiment 
- As discussed below, there are two ways of doing this - 
    * selecting a condition at random using a random number generator 
    * selecting a random order of the stimuli  

### Random Sampling

- In this implementation, I am going to make an experiment that has the number of trials set by the variable **ntrials**.  
- On each trial, I am going to use rng.integers to randomly select whether to play A or B. 
- I am going to assign the integer 1 to A, and the integer 2 to B.  
- First lets consider the logic of randomly selecting A or B 

In [6]:
trial_type  = rng.integers(1,3) # This will randomly select a 1 or 2 
print(trial_type)
if trial_type == 1:  # if trial_type is 1 we are going to play A 
    play_sound(A,volume=volume)
else:
    play_sound(B,volume= volume)

2


- Run the code block above a few times.  We should see that each time we run it, we select a note at random.  

- We could put that block of code in a for loop and repeat for a certain number of *trials*.  I will set the variable **ntrials** to determine the number of trials. 

- I also need to save what i presented on each trial so I know the correct answer. I am making a lit **condition** to hold this information 

In [10]:
ntrials = 6
condition = []
for j in range(ntrials):
    trial_type  = rng.integers(1,3) # This will randomly select a 1 or 2 
    if trial_type == 1:  # if trial_type is 1 we are going to play A 
        play_sound(A,volume=volume)
        condition.append('A')
    else:
        play_sound(B,volume= volume)
        condition.append('B')
print(condition)

['A', 'B', 'A', 'B', 'A', 'A']


- In principle we could make a small task now, where we play a random note and ask the subject to identify it as A or B.  This type of task is known as a **Two-Alternative Forced Choice** task 
- We just have to collect the responses which we discuss in the next section.  

### Random permutation

- One limitation of the randomization approach presented in the previous section is that we dont have control over the number of trials presented in each condition of the experiment.  

- That is, we will have a different number of A and B notes.   

- In many situations, we want to make sure the number of trials per condition is equal.  In that case, we have to use random **permutation** instead of random sampling as our approach.  

In [11]:
trial_1 = np.ones(3) # an array with 3 ones
trial_2 = 2*np.ones(3) # an array with 3 twos
trial_order = np.concatenate((trial_1,trial_2)) # concatenate the array
for j in range(ntrials):
    if trial_order[j] == 1:  # if trial_type is 1 we are going to play A 
        play_sound(A,volume=volume)
    else:
        play_sound(B,volume= volume)
print(trial_order)

[1. 1. 1. 2. 2. 2.]


- That block of code plays the note A three times and then the note B three times.  The order of trials is controlled by the variable trial_order

In order to **randomize** the order of presentation, I need to shuffle the entries of trial_order in a random way. 
The numpy function, `np.random.permutation` can enable us to do this.  

In [12]:
trial_1 = np.ones(3) # an array with 3 ones
trial_2 = 2*np.ones(3) # an array with 3 twos
trial_order = np.concatenate((trial_1,trial_2)) # concatenate the array
trial_order = np.random.permutation(trial_order)  # random permutation of the array
print(trial_order)
condition = list()
for j in range(ntrials):
    if trial_order[j] == 1:  # if trial_type is 1 we are going to play A 
        play_sound(A,volume=volume)
        condition.append('A')
    else:
        play_sound(B,volume= volume)
        condition.append('B')
print(condition)

[1. 2. 2. 2. 1. 1.]
['A', 'B', 'B', 'B', 'A', 'A']


- The advantage of this logic is that we can control the number of trials per condition and equate them.  

### Responses from the Keyboard - `input`

We can obtain responses from the keyboard using the `input` command

You have done this for your Problem Set and Homework.  

I am going to use that function here.  

In [14]:
def get_response(prompt,validresponses):
#    this converts responses to array, comment if using list method
    validresponses = np.array(validresponses)
    invalid = True
    while invalid:
        response = input(prompt)
        if np.any(validresponses == response):
#        if you choose to leave responses as a list then 
#        if response in validresponses:
            print(response)
            invalid = False
        else:
            print('Invalid Response, Try Again')
    return response

- Let's put it together and make an A-B experiment 

In [15]:
ntrials_per_condition = 2 # number of trials per condition
trial_1 = np.ones(ntrials_per_condition) # an array with 2 ones
trial_2 = 2*np.ones(ntrials_per_condition) # an array with 2 twos
trial_order = np.concatenate((trial_1,trial_2)) # concatenate the array
trial_order = random.permutation(trial_order) # get a random order of trials
ntrials = len(trial_order)
condition = list()
trialresponse = list()
for j in range(ntrials):
    if trial_order[j] == 1:  # if trial_type is 1 we are going to play A 
        play_sound(A,volume=volume)
        condition.append('A')
    else:
        play_sound(B,volume= volume)
        condition.append('B')
    response = get_response('Which note was that? (A or B): ', ['A','B'])
    trialresponse.append(response)
print('DONE!')

Which note was that? (A or B):  A


A


Which note was that? (A or B):  A


A


Which note was that? (A or B):  B


B


Which note was that? (A or B):  B


B
DONE!


How do I know how well I did? 


In [16]:
print(condition)

['B', 'B', 'A', 'A']


In [17]:
print(trialresponse)

['A', 'A', 'B', 'B']


So, if I compare the condition variable to the trialresponse variable, I could tell when they get it right 

In [29]:
print(condition == trialresponse)

False


Lists dont compare item by item.  Arrays do! 

In [18]:
print(np.array(condition) == np.array(trialresponse))

[False False False False]


In [19]:
accuracy = np.sum(np.array(condition) == np.array(trialresponse)) / ntrials
print('Your accuracy was: ', accuracy*100, '%')

Your accuracy was:  0.0 %
